In [2]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

from Frame import Frame
import Utils as Utils
import numpy as np
import matplotlib.pyplot as plt
import Plotters
from plyfile import PlyData

import pickle
import numpy as np
import plotly.graph_objects as go

import matplotlib.pyplot as plt
import Utils
%matplotlib qt


path = 'C:/Users/Roni/Documents/gs_input/frames_model.pkl'


# dict_path = 'D:/Documents/data_for_gs/fly_gray/dict/frames_model.pkl'








path_output = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'
# path_output = 'D:/Documents/gaussian_model_output/'

# dict_path  = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/frames_model.pkl'
# image_path = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'
model_name = 'fly_to_bee'
file_name = 'bee_model_dense_10000'

model_name = 'fly_model_to_fly'
file_name = 'fly_model'

dict_path  = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'
image_path = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'
# model_name = 'only_fly'
# file_name = 'fly_model'

# model_name = 'model_8_4_25_deform_rec'
# file_name = 'model_rotation_lr_center0.07_densify_grad_threshold_0.00035'

# model_name = 'model_run'
# dict_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/frames_model.pkl'
# image_path =  'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'
# model_name = '3dgs_bee'
# file_name = 'bee_try'


path_angles = f'{path_output}/{model_name}/{file_name}_angles.pkl'
path_results = f'{path_output}/{model_name}/{file_name}_angles.pkl'


# download model_run localy
path = f'D:/Documents/gaussian_model_output/{model_name}/{file_name}.pkl'
if os.path.exists(f'{path}'):
    with open(path, 'rb') as handle:
        output_angles_weights = pickle.load(handle)

iteration = 1200

frame0 = 1430
frame_end = 1447
weight_flag = False

with open(dict_path,'rb') as f:
    frames = pickle.load(f)

vertices_list = []
image_list = []
weights_list = []
gaussian_list = []
idx_parts = []
xyz_rotated = []
for frame in range(frame0,frame_end):
    ew_to_lab = frames[frame][1][list(frames[frame][1].keys())[0]]['ew_to_lab']
    input_dir = f'{path_output}/{model_name}/{file_name}'
    input_file = f'{path_output}/{model_name}/{frame}/{file_name}/point_cloud/iteration_{iteration}/point_cloud.ply'
    vertices = PlyData.read(input_file)["vertex"]
    xyz = np.column_stack((vertices['x'],vertices['y'],vertices['z']))
    vertices_list.append(xyz)
    xyz_rotated.append((ew_to_lab @ xyz.T).T)

    frames_per_cam = [Frame(image_path,frame,cam, frames_dict = frames)  for cam in range(4)]
    image_list.append(frames_per_cam)
    if os.path.exists(f'{input_dir}_results.pkl'):
        with open(f'{input_dir}_results.pkl', 'rb') as handle:
            output_angles_weights = pickle.load(handle)
        weights_list.append(output_angles_weights['weights'])
        weight_flag = True
        idx_parts.append([np.sum(output_angles_weights['weights'][frame - frame0][iteration][:,idx:idx + 3],axis = 1) == 1 for idx in range(0,9,3)])
        color_list = ['lime','crimson','dodgerblue']
        color_list_2d = ['lime','crimson','dodgerblue']


# frames_per_cam = [Frame(image_path,frame,cam_num, frames_dict = frames) for cam_num in range(4)]


FileNotFoundError: [Errno 2] No such file or directory: 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'

In [6]:
from Frame_Pose import Frame_Pose 
frame_pose = Frame_Pose(xyz_rotated,frame,idx_parts,frame0)

IndexError: boolean index did not match indexed array along dimension 0; dimension is 4247 but corresponding boolean dimension is 4517

In [ ]:

xax_frames = np.vstack([frame_pose[frame].xbody for frame in range(0,frame_end - frame0)])
left_span_frames = np.vstack([frame_pose[frame].left_wing_span for frame in range(0,frame_end - frame0)])

idx_sign =np.where(np.diff(np.sign(np.sum(xax_frames*left_span_frames,axis = 1))) ==2)[0]
left_span_frames[idx_sign[0],:]




array([-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.])

In [4]:
from Frame_Pose import Frame_Pose 

Frame_Pose(xyz_rotated,1447,idx_parts,frame0)



IndexError: boolean index did not match indexed array along dimension 0; dimension is 4247 but corresponding boolean dimension is 4517

In [4]:
import plotly.graph_objects as go

t = np.linspace(-0.001, 0.001, 100)  # Small range since your data seems very small-scale

r_line_points = frame_pose.right_wing_origin + t[:, np.newaxis] * frame_pose.right_wing_direction
l_line_points = frame_pose.left_wing_origin + t[:, np.newaxis] * frame_pose.left_wing_direction

fig = go.Figure()
Plotters.scatter3d(fig,frame_pose.right_wing_xyz,'red',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,frame_pose.left_wing_xyz,'blue',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,frame_pose.body_xyz,'green',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,frame_pose.right_wing_le,'black',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,r_line_points,'orange',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_line_points,'orange',3,'wing',show_colorbar = False)


Plotters.scatter3d(fig, np.vstack((np.mean(frame_pose.right_wing_xyz,axis = 0),np.mean(frame_pose.right_wing_xyz,axis = 0) + frame_pose.right_wing_span*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(frame_pose.left_wing_xyz,axis = 0),np.mean(frame_pose.left_wing_xyz,axis = 0) + frame_pose.left_wing_span*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(frame_pose.body_xyz,axis = 0),np.mean(frame_pose.body_xyz,axis = 0) + frame_pose.xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
fig.show()

In [ ]:
frame_pose

array([ 0.01322226, -0.00535084, -0.00582356])

In [ ]:
import numpy as np
from scipy.linalg import svd
import pandas as pd
import plotly.graph_objects as go
from skimage.measure import LineModelND, ransac


def get_principle_axes(frame_xyz):
    body_cm = np.mean(frame_xyz,axis = 0)
    body_centered = frame_xyz - body_cm
    U, S, Vt = svd(body_centered, full_matrices=False)
    return Vt

def get_axis_orientation(axis,points_from,points_to):
    direction = (np.mean(points_to,axis = 0) - points_from)/np.linalg.norm(np.mean(points_to,axis = 0) - points_from)
    return -axis if np.dot(direction,axis) < 0 else axis


def reorient_axis(points,direction,percent = 0.2):
    projected_on_body = np.dot(points,direction)
    min_points = min(projected_on_body)
    max_points = max(projected_on_body)
    perc_of_body_length = (max_points - min_points)*percent
    bottom = points[(projected_on_body  < (min_points + perc_of_body_length)),:]
    top = points[(projected_on_body  > (max_points - perc_of_body_length)),:]
    x_ax = np.mean(top,axis = 0) - np.mean(bottom,axis = 0)
    return x_ax/np.linalg.norm(x_ax),bottom,top
    

def get_wing_le(xyz,span):

    projected_on_span = np.dot(xyz,span)

    half_wing = 0.7*(max(projected_on_span) - min(projected_on_span))
    xyz_for_le = xyz[projected_on_span < (min(projected_on_span) + half_wing),:]
    projected_on_span = np.dot(xyz_for_le,span)
    projected_on_chord = np.dot(xyz_for_le,chord_rw)


    diff = (max(projected_on_span) - min(projected_on_span))/100
    bin_edges = np.arange(np.min(projected_on_span), np.max(projected_on_span) + diff, diff)
    bin_indices = np.digitize(projected_on_span, bins=bin_edges)
    real_indices = np.array(range(len(projected_on_chord)))
    coord = []
    for idx in bin_indices:
        max_of_bin = np.argmax(projected_on_chord[bin_indices == idx])
        real_idx = real_indices[bin_indices == idx][max_of_bin]
        coord.append(xyz_for_le[real_idx,:])

    return np.vstack(coord)


def ransac_for_le(wing_le):
    
    model_robust, inliers = ransac(wing_le, LineModelND, min_samples=2, residual_threshold=5/100000, max_trials=1000
    )
    origin, direction = model_robust.params
    return origin, direction



frame = 1430
def get_body_x_and_le(frame,frame0,xyz_rotated):
    frame_idx = idx_parts[frame - frame0]
    body_xyz = xyz_rotated[frame - frame0][frame_idx[0],:]
    r_wing_xyz = xyz_rotated[frame - frame0][frame_idx[1],:]
    l_wing_xyz = xyz_rotated[frame - frame0][frame_idx[2],:]

    body_cm = np.mean(body_xyz,axis = 0)
    xbody = get_principle_axes(body_xyz)[0]
    xbody = get_axis_orientation(xbody,[[0,0,0]],[[0,0,1]])

    rwing_axes = get_principle_axes(r_wing_xyz)
    lwing_axes = get_principle_axes(l_wing_xyz)

    span_rw = get_axis_orientation(rwing_axes[0],body_cm,r_wing_xyz)
    span_lw = get_axis_orientation(rwing_axes[0],body_cm,l_wing_xyz)

    chord_rw = get_axis_orientation(rwing_axes[1],[[0,0,0]],[[0,0,1]])
    chord_lw = get_axis_orientation(rwing_axes[1],[[0,0,0]],[[0,0,1]])

    xbody,bottom,top = reorient_axis(body_xyz,xbody,percent = 0.2)


    r_wing_le = get_wing_le(r_wing_xyz,span_rw)
    l_wing_le = get_wing_le(l_wing_xyz,span_lw)


    r_wing_origin, r_wing_direction = ransac_for_le(r_wing_le)
    l_wing_origin, l_wing_direction = ransac_for_le(l_wing_le)



In [17]:

t = np.linspace(-0.001, 0.001, 100)  # Small range since your data seems very small-scale

r_line_points = r_wing_origin + t[:, np.newaxis] * r_wing_direction
l_line_points = l_wing_origin + t[:, np.newaxis] * l_wing_direction

fig = go.Figure()
Plotters.scatter3d(fig,r_wing_xyz,'red',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_wing_xyz,'blue',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,body_xyz,'green',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,r_wing_le,'black',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,r_line_points,'orange',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_line_points,'orange',3,'wing',show_colorbar = False)


Plotters.scatter3d(fig, np.vstack((np.mean(r_wing_xyz,axis = 0),np.mean(r_wing_xyz,axis = 0) + span_rw*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(l_wing_xyz,axis = 0),np.mean(l_wing_xyz,axis = 0) + span_lw*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(body_xyz,axis = 0),np.mean(body_xyz,axis = 0) + xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
fig.show()


In [ ]:
r_wing_direction

l_wing_direction

array([ 0.93693393, -0.32690911,  0.12363352])

In [53]:


t = np.linspace(-0.001, 0.001, 100)  # Small range since your data seems very small-scale

model_robust, inliers = ransac(
    le_points, LineModelND, min_samples=2, residual_threshold=5/100000, max_trials=1000
)
origin, direction = model_robust.params

line_points = origin + t[:, np.newaxis] * direction


outliers = inliers == False

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(
    le_points[inliers][:, 0],
    le_points[inliers][:, 1],
    le_points[inliers][:, 2],
    c='b',
    marker='o',
    label='Inlier data',
)
ax.scatter(
    le_points[outliers][:, 0],
    le_points[outliers][:, 1],
    le_points[outliers][:, 2],
    c='r',
    marker='o',
    label='Outlier data',
)
ax.legend(loc='lower left')



# Plot the fitted line
ax.plot(
    line_points[:, 0],
    line_points[:, 1],
    line_points[:, 2],
    c='k',
    linewidth=2,
    label='Fitted line'
)
plt.show()

In [249]:
fig = go.Figure()
Plotters.scatter3d(fig,r_wing_xyz,'red',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_wing_xyz,'blue',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,body_xyz,'green',3,'wing',show_colorbar = False)


Plotters.scatter3d(fig, np.vstack((np.mean(r_wing_xyz,axis = 0),np.mean(r_wing_xyz,axis = 0) + r_xwing*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(l_wing_xyz,axis = 0),np.mean(l_wing_xyz,axis = 0) + l_xwing*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(body_xyz,axis = 0),np.mean(body_xyz,axis = 0) + xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
fig.show()

In [233]:
r_xwing_orient,bottom,top = reorient_axis(r_wing_xyz,r_xwing,percent = 0.1)
arrow = np.vstack((np.mean(bottom,axis = 0),np.mean(bottom,axis = 0) + r_xwing*3/1000))

fig = go.Figure()
Plotters.scatter3d(fig, r_wing_xyz,'green',3,'x') 
Plotters.scatter3d(fig, np.vstack((np.mean(bottom,axis = 0),np.mean(bottom,axis = 0) + r_xwing*4/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, bottom,'red',3,'x') 
Plotters.scatter3d(fig, top,'red',3,'x') 
Plotters.scatter3d(fig, arrow,'blue',3,'x',mode = 'markers+lines') 
fig.show()